# ML Experimentation with Real-Time Inference & Model Monitoring

## Overview
This notebook demonstrates an end-to-end machine learning workflow with [Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html) that includes:

- **Model Training**: Train an XGBoost model using SageMaker [ModelTrainer (SDK v3)](https://sagemaker.readthedocs.io/en/stable/training/index.html)
- **MLflow Integration**: Track experiments and log artifacts to [SageMaker AI MLflow App](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- **Real-Time Endpoint**: Deploy a SageMaker [real-time endpoint](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html) with [data capture](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture.html) enabled
- **Data Drift Monitoring**: Use [Evidently (open-source)](https://www.evidentlyai.com/evidently-oss) to detect data drift on captured data
- **Model Quality Monitoring**: Use Evidently (open-source) to generate a model quality report
- **Drift Metrics in MLflow**: Log drift metrics and reports to MLflow for tracking

## Workflow
The diagram below shows the full monitoring workflow using a real-time inference endpoint.

1. Train a model in a [training job](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-training.html), storing the baseline data set for the trained model in S3.
2. Deploy the model in a real-time endpoint with data capture enabled; data capture stores the input and output from the endpoint in S3.
3. Use Evidently to calculate data drift and model quality based on the baseline dataset and the data captured from the endpoint.
    1. In the diagram, the Evidently code runs in a [processing job](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job.html) inside a [pipeline](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html). This enables running the drift calculations based on a schedule or based on a trigger from S3. For simplicity, this notebook runs the Evidently calculations locally.
4. Save the data drift and model quality reports to MLflow for tracking and comparison.
5. Optionally, trigger an alert if data or model drift exceeds a certain threshold. The alert can be raised directly in Slack through [Amazon SNS](https://docs.aws.amazon.com/sns/latest/dg/welcome.html).

![Model monitoring workflow with real-time endpoints](img/arch-sagemaker-inference-predictiveml-monitoring-RealTime-inf.png "Model monitoring workflow with real-time endpoints")

### Business Problem
This sample uses the [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) dataset from the UCI Machine Learning Repository. It can be used to predict whether a bank customer will subscribe to a term deposit based on:
- Demographics (age, job, marital status, education)
- Financial information (credit default, housing loan, personal loan)
- Campaign data (contact type, month, day of week, duration)
- Economic indicators (employment rate, consumer price index, etc.)

---

<div class="alert alert-warning" role="alert">
<h3>⚠️ SIMULATED PRODUCTION DATA WITH ARTIFICIAL DRIFT</h3>
<p><strong>Purpose:</strong> Use the notebook to demonstrate drift detection capabilities, not for production deployments</p>

## 1: Prerequisites & Setup

Install required packages and import libraries. This may take 1-2 minutes on first run.

In [ ]:
# %pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
# %pip install --no-cache-dir sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops 'sagemaker>=3' mlflow==3.4.0 sagemaker-mlflow==0.2.0
# %pip install -Uq "pandas" "scikit-learn" "xgboost" "evidently>=0.7.20" --no-cache-dir --ignore-installed

In [1]:
%pip install "evidently>=0.7.20"

  Using cached iterative_telemetry-0.0.10-py3-none-any.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 84.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 192.9 MB/s  0:00:00
Using cached iterative_telemetry-0.0.10-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.3/568.3 kB 73.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 134.5 MB/s  0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 6.0.1
    Uninstalling plotly-6.0.1:0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [plotly]
      Successfully uninstalled plotly-6.0.1━━━━━━━━━━━━━━━━━━━━━━━  2/14 [plotly]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [evidently]14 [evidently]]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dash 2.18.1 requires dash-core-components==2.0.0, which is 

In [1]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

Stored variables and their in-db values:
baseline_s3_url                             -> 's3://sagemaker-us-west-2-975049911976/from-idea-t
bucket_name                                 -> 'sagemaker-us-west-2-975049911976'
bucket_prefix                               -> 'from-idea-to-prod/xgboost'
dataset_feature_group_name                  -> 'from-idea-to-prod-31-21-10-19'
dataset_file_local_path                     -> 'data/bank-additional/bank-additional-full.csv'
domain_id                                   -> 'd-batsamo3abdt'
evaluation_s3_url                           -> 's3://sagemaker-us-west-2-975049911976/from-idea-t
experiment_name                             -> 'from-idea-to-prod-experiment--260413-0713'
feature_store_bucket_prefix                 -> 'from-idea-to-prod/feature-store'
image_arn                                   -> 'arn:aws:sagemaker:us-west-2:542918446943:image/sa
image_version_alias                         -> '3.9.0'
initialized                                

In [2]:
import boto3
import sagemaker
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import zipfile
import os
import json
import time
from datetime import datetime

from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import SourceCode, InputData, Compute
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.core.transformer import Transformer

from evidently import Report, Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset
from evidently.metrics import *

print(f'MLflow version: {mlflow.__version__}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Fetched defaults config from location: /home/sagemaker-user/.config/sagemaker/config.yaml
MLflow version: 3.4.0


In [3]:
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = 'bank-marketing-monitoring'

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  S3 Bucket: {bucket}')
print(f'  IAM Role: {role}')
print(f'  Data Prefix: {prefix}')

AWS Configuration:
  Region: us-west-2
  S3 Bucket: sagemaker-us-west-2-975049911976
  IAM Role: arn:aws:iam::975049911976:role/from-idea-to-prod-SageMakerExecutionRole-h5rrBhGSjbNO
  Data Prefix: bank-marketing-monitoring


---

## 2: SageMaker AI MLflow App Configuration

### What is a SageMaker AI MLflow App?

A fully managed service based on open-source MLflow that provides:
- **Experiment Tracking**: Log parameters, metrics, and artifacts
- **Model Registry**: Version and manage models
- **SageMaker Integration**: Automatic registration to SageMaker AI Model Registry
- **Reproducibility**: Track code versions and dependencies
- **Collaboration**: Share experiments across teams

### Instructions

1. In SageMaker Studio, navigate to the left sidebar
2. Click on "MLflow" panel under "Applications"
3. Find your MLflow app (usually named "DefaultMLFlowApp") or create one if necessary
4. Copy the app name and edit it in the cell below if you need to use a specific app

The cell below checks if the MLflow App exists and will fetch the corresponding ARN.

In [4]:
# mlflow_app_name = 'DefaultMLFlowApp'

# sm_client = boto3.client('sagemaker', region_name=region)
# mlflow_list = sm_client.list_mlflow_apps()

# print(f'Found {len(mlflow_list["Summaries"])} MLflow app(s) in your account:')
# for app in mlflow_list['Summaries']:
#     print(f'  - {app["Name"]}')

# mlflow_app_arn = None
# for mlflow_app in mlflow_list['Summaries']:
#     if mlflow_app['Name'] == mlflow_app_name:
#         mlflow_app_arn = mlflow_app['Arn']
#         break

# if mlflow_app_arn:
#     print(f'\n Using MLflow app: {mlflow_app_name}')
#     print(f'  ARN: {mlflow_app_arn}')
# else:
#     raise ValueError(f'MLflow app "{mlflow_app_name}" not found. Please check the name or create one in SageMaker Studio.')

The ARN fetched by the previous cell is set as the default MLflow tracking URI. A new experiment is created within MLflow if required, otherwise the existing experiment is set as the default.

In [5]:
mlflow.set_tracking_uri(mlflow_arn)
mlflow_experiment_name = 'demo-ml-xgb-train-monitor'
mlflow_model_name = 'xgb-ml'

try:
    # Try to create a new experiment
    experiment_id = mlflow.create_experiment(mlflow_experiment_name)
    print(f'Created new MLflow experiment: {mlflow_experiment_name}')
    print(f'Experiment ID: {experiment_id}')
except:
    # Experiment already exists, set it as active
    mlflow.set_experiment(mlflow_experiment_name)
    experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
    print(f'Using existing MLflow experiment: {mlflow_experiment_name}')
    print(f'Experiment ID: {experiment.experiment_id}')

Using existing MLflow experiment: demo-ml-xgb-train-monitor
Experiment ID: 83


---

## 3: Data Preparation
This section downloads the bank marketing dataset and applies simple preprocessing steps to prepare the dataset for training an XGBoost model.

First, download and extract the data from the UCI Machine Learning Repository.

In [6]:
print('Downloading bank marketing dataset...')
!wget -N https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
!unzip -o bank-additional.zip
print('\nDataset downloaded and extracted')

--2026-04-15 15:27:06--  https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘bank-additional.zip’

bank-additional.zip     [ <=>                ] 434.15K  --.-KB/s    in 0.1s    

Last-modified header missing -- time-stamps turned off.
2026-04-15 15:27:06 (3.10 MB/s) - ‘bank-additional.zip’ saved [444572]

Archive:  bank-additional.zip
  inflating: bank-additional/.DS_Store  
  inflating: __MACOSX/bank-additional/._.DS_Store  
  inflating: bank-additional/.Rhistory  
  inflating: bank-additional/bank-additional-full.csv  
  inflating: bank-additional/bank-additional-names.txt  
  inflating: bank-additional/bank-additional.csv  
  inflating: __MACOSX/._bank-additional  

Dataset downloaded and extracted


Load the data into a pandas DataFrame and do a quick check on the target distribution.

In [7]:
df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')
print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['y'].value_counts())
print(f'\nFirst few rows:')
df.head()

Dataset shape: (41188, 21)

Target distribution:
y
no     36548
yes     4640
Name: count, dtype: int64

First few rows:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


Encode all of the categorical features as well as the target variable.

In [8]:
cat_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns if c != 'y']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

df['y'] = (df['y'] == 'yes').astype(int)
print(f'Encoded {len(cat_cols)} categorical features')
print(f'Feature names: {df.columns.tolist()}')

Encoded 10 categorical features
Feature names: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


Now split the data into training (70%), validation (15%), and test (15%) sets with stratified sampling.

The training data set is saved twice: once for training the model, and once more to serve as the baseline data set for calculating data drift.

In [9]:
X, y = df.drop('y', axis=1), df['y']

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'Training set: {X_train.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Test set: {X_test.shape}')

os.makedirs('data', exist_ok=True)
pd.concat([y_train, X_train], axis=1).to_csv('data/train.csv', index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv('data/validation.csv', index=False, header=False)
pd.concat([y_test, X_test], axis=1).to_csv('data/test.csv', index=False, header=False)

# Save baseline (reference) data for drift monitoring - use training data features
X_train.to_csv('data/baseline.csv', index=False, header=True)
print('Data split and saved locally')

Training set: (28831, 20)
Validation set: (6178, 20)
Test set: (6179, 20)
Data split and saved locally


Upload all of the data sets to S3.

In [10]:
train_s3 = sagemaker_session.upload_data('data/train.csv', bucket, f'{prefix}/data/train')
validation_s3 = sagemaker_session.upload_data('data/validation.csv', bucket, f'{prefix}/data/validation')
test_s3 = sagemaker_session.upload_data('data/test.csv', bucket, f'{prefix}/data/test')
baseline_s3 = sagemaker_session.upload_data('data/baseline.csv', bucket, f'{prefix}/data/baseline')

print(f'Train S3: {train_s3}')
print(f'Validation S3: {validation_s3}')
print(f'Test S3: {test_s3}')
print(f'Baseline S3: {baseline_s3}')

Train S3: s3://sagemaker-us-west-2-975049911976/bank-marketing-monitoring/data/train/train.csv
Validation S3: s3://sagemaker-us-west-2-975049911976/bank-marketing-monitoring/data/validation/validation.csv
Test S3: s3://sagemaker-us-west-2-975049911976/bank-marketing-monitoring/data/test/test.csv
Baseline S3: s3://sagemaker-us-west-2-975049911976/bank-marketing-monitoring/data/baseline/baseline.csv


---
## 4: Model Training with tracking in MLflow

Train an [XGBoost model](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html) using SageMaker ModelTrainer with MLflow integration.

### Key Features:
- **ModelTrainer**: Simplified SageMaker SDK v3 API for training
- **MLflow Autologging**: Automatically captures parameters, metrics, and model artifacts
- **Model Registry**: Automatic registration to MLflow and SageMaker Model Registry

Create a directory to store the training script and the requirements file.

In [11]:
os.makedirs('scripts', exist_ok=True)

In [12]:
%%writefile scripts/train.py
import argparse
import os
import json
import logging
import sys
import xgboost as xgb
import pandas as pd
import mlflow
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.addHandler(logging.StreamHandler(sys.stdout))


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--max_depth', type=int, default=5)
    parser.add_argument('--eta', type=float, default=0.2)
    parser.add_argument('--gamma', type=int, default=4)
    parser.add_argument('--min_child_weight', type=int, default=6)
    parser.add_argument('--subsample', type=float, default=0.8)
    parser.add_argument('--num_round', type=int, default=100)
    return parser.parse_known_args()

if __name__ == '__main__':
    args, _ = parse_args()
    
    train_data = pd.read_csv('/opt/ml/input/data/train/train.csv', header=None)
    val_data = pd.read_csv('/opt/ml/input/data/validation/validation.csv', header=None)
    
    X_train, y_train = train_data.iloc[:, 1:], train_data.iloc[:, 0]
    X_val, y_val = val_data.iloc[:, 1:], val_data.iloc[:, 0]
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    mlflow_app_arn = os.environ.get('MLFLOW_TRACKING_URI', None)
    mlflow_experiment_name = os.environ.get('MLFLOW_EXP', None)
    mlflow_model_name = os.environ.get('MLFLOW_MODEL_NAME', None)
    mlflow.set_tracking_uri(mlflow_app_arn)
    mlflow.set_experiment(mlflow_experiment_name)
    
    mlflow.xgboost.autolog(
        log_input_examples=True,
        log_model_signatures=True,
        log_models=True,
        log_datasets=True,
        model_format="json",
        registered_model_name=mlflow_model_name,
        extra_tags={"team": "data-science", "use_case": "bank-marketing"},
    )
    
    with mlflow.start_run(run_name=f"training_{mlflow_model_name}"):
        params = {
            'max_depth': args.max_depth,
            'eta': args.eta,
            'gamma': args.gamma,
            'min_child_weight': args.min_child_weight,
            'subsample': args.subsample,
            'objective': 'binary:logistic',
            'eval_metric': 'auc'
        }
        
        mlflow.log_params(params)
        model = xgb.train(params, dtrain, args.num_round, evals=[(dval, 'validation')])
        
        y_pred_proba = model.predict(dval)
        y_pred = (y_pred_proba > 0.5).astype(int)
        
        metrics = {
            'Accuracy': accuracy_score(y_val, y_pred),
            'Precision': precision_score(y_val, y_pred),
            'Recall': recall_score(y_val, y_pred),
            'F1Score': f1_score(y_val, y_pred),
            'auc': roc_auc_score(y_val, y_pred_proba)
        }
        
        mlflow.log_metrics(metrics)
        print(f'Validation Metrics: {metrics}')
        
        model_path = '/opt/ml/model'
        os.makedirs(model_path, exist_ok=True)
        model.save_model(f'{model_path}/{mlflow_model_name}')
        mlflow.xgboost.log_model(
            model, 
            artifact_path="model",
            registered_model_name=mlflow_model_name
        )

Overwriting scripts/train.py


In [13]:
requirements = '''mlflow==3.1.4
sagemaker-mlflow==0.2.0
scikit-learn
pandas
'''

with open('scripts/requirements.txt', 'w') as f:
    f.write(requirements)
    
print('Requirements file created')

Requirements file created


SageMaker provides pre-built containers for various frameworks, including the [XGBoost container](https://github.com/aws/sagemaker-xgboost-container). The requirements file created previously will be used to install some additional packages on the pre-built container, which is fetched below.

In [14]:
# xgboost_image = image_uris.retrieve(
#     framework='xgboost',
#     region=region,
#     version='1.7-1',
#     image_scope='training',
# )

xgboost_image
print(f'XGBoost training image: {xgboost_image}')



XGBoost training image: 975049911976.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost-custom:3.0-5


Use ModelTrainer to set up the SageMaker training job with the training script created previously, the pre-built container, a specific compute configuration, and a set of hyperparameters. MLflow configuration is passed through environment variables so the training job can log metrics and parameters into the MLflow App.

In [16]:
source_code = SourceCode(
    source_dir='scripts',
    entry_script='train.py',
    # requirements="requirements.txt"
)

compute = Compute(
    instance_type='ml.r5.xlarge',
    instance_count=1,
    volume_size_in_gb=30
)

hyperparameters = {
    'max_depth': 5,
    'eta': 0.2,
    'gamma': 4,
    'min_child_weight': 6,
    'subsample': 0.8,
    'num_round': 100
}

model_trainer = ModelTrainer(
    training_image=xgboost_image,
    source_code=source_code,
    compute=compute,
    hyperparameters=hyperparameters,
    base_job_name='bank-marketing-xgboost',
    environment={
        'MLFLOW_TRACKING_URI': mlflow_arn,
        'MLFLOW_EXP': mlflow_experiment_name,
        'MLFLOW_MODEL_NAME': mlflow_model_name
    }
)

[04/15/26 15:27:31] INFO     SageMaker session not provided. Using default Session.                  ]8;id=631280;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=904262;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Role not provided. Using default role:                                  ]8;id=187301;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=563712;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::975049911976:role/from-idea-to-prod-SageMakerExecutionRole               
                             -h5rrBhGSjbNO                                                                         

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=341972;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=837975;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=686477;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=848031;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-975049911976/bank-marketing-x                
                             gboost' kms_key_id=None compression_type='GZIP'                                       
                             remove_job_name_from_s3_output_path=Unassigned()                                      
                             disable_model_upload=Unassigned() channels=Unassigned()                               

                    INFO     Training image URI:                                               ]8;id=651019;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=112395;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#553\553]8;;\
                             975049911976.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost-cu                     
                             stom:3.0-5                                                                            

Start the training job with the training and validation datasets. It will take approximately 5-7 minutes to complete. 

In [17]:
%%time
input_data_train = InputData(channel_name='train', data_source=train_s3)
input_data_validation = InputData(channel_name='validation', data_source=validation_s3)

print('Starting model training...')

model_trainer.train(
    input_data_config=[input_data_train, input_data_validation],
    wait=True
)

training_job_name = model_trainer.base_job_name
print(f'\n Training completed: {training_job_name}')

Starting model training...


[04/15/26 15:27:37] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=247451;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=241018;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#100\100]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Fetched defaults config from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[04/15/26 15:27:39] INFO     Creating training_job resource.                                     ]8;id=424193;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=617906;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=712811;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=793814;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#356\356]8;;\

...

                    INFO     bank-marketing-xgboost-20260415152738/algo-1-1776266891:            ]8;id=986846;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=250100;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35871\35871]8;;\
                             🧪 View experiment at:                                                                
                             https://mlflow.sagemaker.us-west-2.app.aws/#/experiments/83                           

                    INFO     bank-marketing-xgboost-20260415152738/algo-1-1776266891:            ]8;id=665657;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=198915;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35871\35871]8;;\
                             Training Container Execution Completed                                                

                    INFO     bank-marketing-xgboost-20260415152738/algo-1-1776266891:            ]8;id=291613;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=87032;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35871\35871]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

[04/15/26 15:32:33] INFO     Final Resource Status: Completed                                    ]8;id=95131;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=103755;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35874\35874]8;;\


 Training completed: bank-marketing-xgboost
CPU times: user 4.71 s, sys: 417 ms, total: 5.13 s
Wall time: 4min 56s


Once the training job is finished, you can fetch the location of the trained model artifacts.

In [18]:
model_data_s3 = model_trainer._latest_training_job.model_artifacts.s3_model_artifacts
print(f'Model artifacts location: {model_data_s3}')

Model artifacts location: s3://sagemaker-us-west-2-975049911976/bank-marketing-xgboost/bank-marketing-xgboost-20260415152738/output/model.tar.gz


### View Training Results in MLflow

To view your training results:

1. In SageMaker Studio, click on **MLflow** in the left sidebar
2. Find your MLflow app (e.g. DefaultMLFlowApp)
3. Click to open the MLflow UI
4. Navigate to the experiment: **demo-ml-xgb-train-monitor**
5. View parameters, metrics, and artifacts logged during training

---

## 5: Real-Time Endpoint with Data Capture

Deploy the trained model as a SageMaker real-time endpoint with data capture enabled. Data capture logs the inputs to your endpoint and the inference outputs from your deployed model to Amazon S3. 

### Why Real-Time Endpoints with Data Capture?
- **Low latency**: Sub-second inference for real-time applications
- **Data Capture**: Automatically logs request/response payloads to S3
- **Monitoring**: Captured data enables drift detection and model quality monitoring
- **Production-ready**: Supports auto-scaling, A/B testing, and shadow deployments

Create a directory to store the inference script. In this case, because this notebook uses the pre-built XGBoost container, the inference code only needs a model load method.

In [19]:
os.makedirs('inference_code', exist_ok=True)

In [20]:
%%writefile inference_code/inference.py
import xgboost as xgb
import os

def model_fn(model_dir):
    model_files = [f for f in os.listdir(model_dir) if not f.startswith('.') and f != 'code']
    model = xgb.Booster()
    model.load_model(os.path.join(model_dir, model_files[0]))
    return model

Writing inference_code/inference.py


The pre-built XGBoost container expects the inference code to be packaged together with the model artifacts, but ModelTrainer does not do this automatically. Fortunately, the SDK does provide a `repack_model()` method which helps bundle the inference code into `model.tar.gz`.

In [21]:
from sagemaker.core.common_utils import repack_model

repacked_model_uri = f's3://{bucket}/{prefix}/repacked-model/model.tar.gz'

repack_model(
    inference_script='inference.py',
    source_directory='inference_code',
    dependencies=[],
    model_uri=model_data_s3,
    repacked_model_uri=repacked_model_uri,
    sagemaker_session=sagemaker_session,
)

print(f'Repacked model uploaded to: {repacked_model_uri}')

Repacked model uploaded to: s3://sagemaker-us-west-2-975049911976/bank-marketing-monitoring/repacked-model/model.tar.gz


The repacked model artifacts can now be used with [ModelBuilder](https://sagemaker.readthedocs.io/en/stable/inference/index.html) to create the final Model object ready for deployment. 

In [22]:
from pydantic import ValidationError
from sagemaker.core.model_monitor import DataCaptureConfig

inference_image = image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1',
    image_scope='inference'
)

model_builder = ModelBuilder(
    schema_builder=SchemaBuilder(
        sample_input='1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20',
        sample_output='0.7'
    ),
    image_uri=inference_image,
    s3_model_data_url=repacked_model_uri,
    role_arn=role,
    sagemaker_session=sagemaker_session,
    instance_type='ml.m5.xlarge',
)

built_model = model_builder.build()
model_name = model_builder.model_name
print(f'Model built: {model_name}')

[04/15/26 15:32:34] INFO     Ignoring unnecessary instance type: None.                            ]8;id=613296;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=213113;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=797313;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=867616;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#100\100]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[04/15/26 15:32:35] DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=197254;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=725506;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1323\1323]8;;\
                             MLflow model input                                                                    

[04/15/26 15:32:36] INFO     Creating model with name: model-4b07c9fd                        ]8;id=229695;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=499028;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1814\1814]8;;\

[04/15/26 15:32:37] INFO     ✅ Model has been created: 'model-4b07c9fd' using server None in ]8;id=472947;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=621901;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#3341\3341]8;;\
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-west-2:975049911976:model/model-4b07c9fd)                        

Model built: model-4b07c9fd


A Model object created with ModelBuilder can be deployed multiple times using either real-time endpoints, asynchronous endpoints, serverless inference, or batch transform. In this case, the model is deployed as a real-time endpoint with data capture enabled. Data capture can be used to save either the input, or the output, or both. It can also use sampling to capture a subset of the input/output for monitoring.

In [23]:
%%time
capture_s3_uri = f's3://{bucket}/{prefix}/data-capture'

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=capture_s3_uri,
    capture_options=['Input', 'Output'],
    csv_content_types=['text/csv'],
)

try:
    endpoint = model_builder.deploy(
        instance_type='ml.m5.xlarge',
        initial_instance_count=1,
        data_capture_config=data_capture_config,
    )
except ValidationError:
    # SDK bug: Endpoint.get() fails deserializing data_capture_config when kms_key_id is absent.
    # The endpoint is already deployed successfully at this point.
    pass

endpoint_name = model_builder.endpoint_name
print(f'Endpoint deployed: {endpoint_name}')
print(f'Data capture destination: {capture_s3_uri}')

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=571724;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=556811;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#100\100]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[04/15/26 15:32:38] INFO     Creating endpoint-config with name endpoint-79546b04             ]8;id=886678;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=643234;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#985\985]8;;\

[04/15/26 15:32:39] INFO     Creating endpoint with name endpoint-79546b04                   ]8;id=626472;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=539885;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1017\1017]8;;\

Output()

✅ Created endpoint with name endpoint-79546b04

...

169.254.178.2 - - [15/Apr/2026:15:35:29 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"

[2026-04-15:15:35:34:INFO] No GPUs detected (normal if no gpus installed)

169.254.178.2 - - [15/Apr/2026:15:35:34 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"

[04/15/26 15:35:41] INFO     ✅ Deployment successful: Endpoint 'endpoint-79546b04' using     ]8;id=890115;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=858499;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#2671\2671]8;;\
                             None in SAGEMAKER_ENDPOINT mode (ARN:                                                 
                             arn:aws:sagemaker:us-west-2:975049911976:endpoint/endpoint-79546                      
                             b04)                                                                                  

Endpoint deployed: endpoint-79546b04
Data capture destination: s3://sagemaker-us-west-2-975049911976/bank-marketing-monitoring/data-capture
CPU times: user 3.17 s, sys: 289 ms, total: 3.46 s
Wall time: 3min 3s


### Prepare Ground Truth Data & Invoke Endpoint

To calculate model quality (i.e. accuracy, F1 score), ground truth labels must be available to compare to the predicted labels. In production, the ground truth labels are normally collected asynchronously. For example, when a model is used to forecast sales for the next month, the ground truth sales data will only be available after the month has passed. When a model is used to generate recommendations for a retail website, the ground truth might be available within minutes when a user decides to purchase the product or not. The process for collecting ground truth labels depends on the ML use case.

In this sample notebook, the test dataset is used to calculate model quality so the ground truth labels are already available. To replicate a production use case, the ground truth labels are saved separately, to be merged with the inference results later. Each inference input is assigned a unique ID (e.g. `inf-000010`) to enable merging.

In [24]:
test_features = X_test.copy()
test_features.to_csv('data/test_features.csv', index=False, header=False)

ground_truth = {}
for idx, (i, row) in enumerate(test_features.iterrows()):
    inference_id = f'inf-{idx:06d}'
    ground_truth[inference_id] = int(y_test.iloc[idx])

gt_df = pd.DataFrame(list(ground_truth.items()), columns=['inference_id', 'target'])
gt_df.to_csv('data/ground_truth.csv', index=False)
print(f'Ground truth stored for {len(gt_df)} records')

Ground truth stored for 6179 records


Now the test dataset, without the ground truth labels, can be sent to the real-time endpoint for inference.

In [25]:
runtime_client = boto3.client('sagemaker-runtime', region_name=region)

print(f'Sending {len(test_features)} requests to endpoint...')
for idx, (i, row) in enumerate(test_features.iterrows()):
    inference_id = f'inf-{idx:06d}'
    payload = ','.join(str(v) for v in row.values)
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Body=payload,
        InferenceId=inference_id
    )
    response['Body'].read()  # drain the response

print(f'Sent {len(test_features)} requests')

Sending 6179 requests to endpoint...
Sent 6179 requests


It can take a few minutes for the data capture results to arrive in S3. Data Capture stops capturing requests at high levels of disk usage to prevent impact to inference requests. 

In [26]:
import time as _time

s3_client = boto3.client('s3')
capture_prefix = f'{prefix}/data-capture/{endpoint_name}/AllTraffic'

print('Waiting for data capture files in S3 (may take 1-2 minutes)...')
for attempt in range(12):
    resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=capture_prefix, MaxKeys=1)
    if resp.get('KeyCount', 0) > 0:
        print(f'Data capture files found after {(attempt+1)*15}s')
        break
    _time.sleep(15)
else:
    print('No capture files found yet. They may still be buffering.')

# List captured files
resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=capture_prefix)
capture_keys = [obj['Key'] for obj in resp.get('Contents', [])]
print(f'\nFound {len(capture_keys)} capture file(s)')

Waiting for data capture files in S3 (may take 1-2 minutes)...
Data capture files found after 90s

Found 1 capture file(s)


Data capture stores the input and output in a JSONL file. This file can be loaded into a pandas DataFrame for calculating drift later.

In [27]:
import base64

captured_records = []

for key in capture_keys:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    for line in obj['Body'].read().decode('utf-8').strip().split('\n'):
        record = json.loads(line)
        inference_id = record.get('eventMetadata', {}).get('inferenceId', '')

        input_data = record.get('captureData', {}).get('endpointInput', {})
        output_data = record.get('captureData', {}).get('endpointOutput', {})

        raw_input = input_data.get('data', '')
        if input_data.get('encoding') == 'BASE64':
            raw_input = base64.b64decode(raw_input).decode('utf-8')

        raw_output = output_data.get('data', '')
        if output_data.get('encoding') == 'BASE64':
            raw_output = base64.b64decode(raw_output).decode('utf-8')

        captured_records.append({
            'inference_id': inference_id,
            'input': raw_input.strip(),
            'prediction_proba': float(raw_output.strip()) if raw_output.strip() else None
        })

captured_df = pd.DataFrame(captured_records)
captured_df['prediction'] = (captured_df['prediction_proba'] > 0.5).astype(int)

feature_cols = pd.DataFrame(
    captured_df['input'].str.split(',').tolist(),
    columns=test_features.columns
).astype(float)
captured_df = pd.concat([captured_df[['inference_id', 'prediction_proba', 'prediction']], feature_cols], axis=1)

print(f'Parsed {len(captured_df)} captured records with inferenceId')
print(captured_df[['inference_id', 'prediction_proba', 'prediction']].head())

Parsed 6179 captured records with inferenceId
  inference_id  prediction_proba  prediction
0   inf-000000          0.005879           0
1   inf-000001          0.129020           0
2   inf-000002          0.000871           0
3   inf-000003          0.000569           0
4   inf-000004          0.000741           0


---

## 6: Data Drift Monitoring with Evidently

### What is Data Drift?

Data drift occurs when the statistical properties of input features change over time, which can degrade model performance.

### Why Monitor Drift?
- **Early Warning**: Detect issues before they impact business
- **Model Decay**: Understand when models need retraining
- **Data Quality**: Identify data pipeline issues
- **Compliance**: Track model performance for regulatory requirements

### Evidently Features
- **Open-source**: Free and customizable
- **Statistical Tests**: Multiple drift detection methods (KS test, Chi-square, etc.)
- **Visual Reports**: Interactive HTML reports
- **Flexible**: Works with any ML framework
- **MLflow Integration**: Log metrics and artifacts to MLflow

To calculate data drift, first load the baseline dataset saved during model training.

In [28]:
reference_data = pd.read_csv('data/baseline.csv')

print(f'Reference (baseline) data shape: {reference_data.shape}')
print('\nReference data represents the training data distribution')
print('\nReference data statistics:')
print(reference_data.describe())

Reference (baseline) data shape: (28831, 20)

Reference data represents the training data distribution

Reference data statistics:
                age           job       marital    education       default  \
count  28831.000000  28831.000000  28831.000000  28831.00000  28831.000000   
mean      40.011550      3.737054      1.175644      3.75672      0.210086   
std       10.393815      3.600216      0.608602      2.13235      0.407632   
min       17.000000      0.000000      0.000000      0.00000      0.000000   
25%       32.000000      0.000000      1.000000      2.00000      0.000000   
50%       38.000000      2.000000      1.000000      3.00000      0.000000   
75%       47.000000      7.000000      2.000000      6.00000      0.000000   
max       98.000000     11.000000      3.000000      7.00000      2.000000   

            housing          loan       contact         month   day_of_week  \
count  28831.000000  28831.000000  28831.000000  28831.000000  28831.000000   
mean    

<div class="alert alert-warning" role="alert">
<h3>⚠️ SIMULATED PRODUCTION DATA WITH ARTIFICIAL DRIFT</h3>
<p><strong>Purpose:</strong> Introducing controlled data drift to demonstrate drift detection capabilities</p>

<b>Important:</b> Introducing ARTIFICIAL data drift into the test set for demo purposes only. Do NOT use this in production. In a real scenario, this would be actual production data.

In [29]:
feature_columns = [c for c in captured_df.columns if c not in ['inference_id', 'prediction_proba', 'prediction']]
production_data = captured_df[feature_columns].copy()

if 'age' in production_data.columns:
    production_data['age'] = production_data['age'] + 2

if 'duration' in production_data.columns:
    production_data['duration'] = production_data['duration'] * 1.15

if 'campaign' in production_data.columns:
    production_data['campaign'] = production_data['campaign'] + np.random.normal(0, 0.2, len(production_data))

print(f'Production data shape: {production_data.shape}')
print('\nProduction data statistics:')
print(production_data.describe())
print('\nSimulated drift introduced for demonstration')

Production data shape: (6179, 20)

Production data statistics:
               age          job      marital    education      default  \
count  6179.000000  6179.000000  6179.000000  6179.000000  6179.000000   
mean     42.112154     3.739926     1.160058     3.758861     0.205373   
std      10.582893     3.589558     0.609983     2.145286     0.404007   
min      19.000000     0.000000     0.000000     0.000000     0.000000   
25%      34.000000     0.000000     1.000000     2.000000     0.000000   
50%      40.000000     2.000000     1.000000     3.000000     0.000000   
75%      49.000000     7.000000     2.000000     6.000000     0.000000   
max     100.000000    11.000000     3.000000     7.000000     1.000000   

           housing         loan      contact        month  day_of_week  \
count  6179.000000  6179.000000  6179.000000  6179.000000  6179.000000   
mean      1.069267     0.329665     0.362842     4.249231     2.010520   
std       0.985518     0.725687     0.480859    

Evidently has various presets for calculating different types of [data drift](https://docs.evidentlyai.com/metrics/preset_data_drift), all of which can be customized to suit the particular ML use case. In addition to saving the final report in MLflow, it can be helpful to extract specific drift values and save them separately as metrics in MLflow. This enables you to easily compare different runs or generate alerts based on certain values. The cell below contains a helper function which extracts certain metrics from the Evidently results to log in MLflow. It can be customized depending on the types of data drift calculations you use.

In [30]:
import numpy as np

def log_metrics_from_dict(d):
    metrics = d.get("metrics", []) or []

    drifted_columns_logged = False

    for m in metrics:
        metric_name = m.get("metric_name", "")
        cfg = m.get("config", {}) or {}
        val = m.get("value")

        # 1) DriftedColumnsCount: always log
        if metric_name.startswith("DriftedColumnsCount"):
            drifted_columns_logged = True
            # value is a dict: {"count": ..., "share": ...}
            if isinstance(val, dict):
                count = float(val.get("count", 0.0))
                share = float(val.get("share", 0.0))
            else:
                # Fallback if structure changes
                count = float(val) if val is not None else 0.0
                share = 0.0

            mlflow.log_metric("DriftedColumnsCount.count", count)
            mlflow.log_metric("DriftedColumnsCount.share", share)
            continue

        # 2) ValueDrift: log only if value > threshold
        if metric_name.startswith("ValueDrift"):
            threshold = cfg.get("threshold")
            column = cfg.get("column")

            if threshold is None or column is None:
                continue

            try:
                numeric_val = float(val)
            except (TypeError, ValueError):
                continue

            if numeric_val > float(threshold):
                key = f"ValueDrift:{column}"
                mlflow.log_metric(key, numeric_val)

    # 3) If there was no DriftedColumnsCount metric at all, log 0
    if not drifted_columns_logged:
        mlflow.log_metric("DriftedColumnsCount", 0.0)

In addition to extracting certain metrics, the entire reports produced by Evidently will be saved in a separate folder.

In [31]:
os.makedirs('reports', exist_ok=True)

This sample uses the built-in `DataDriftPreset` and `DataSummaryPreset` from Evidently to calculate data drift from the baseline data and the inference results. The metrics are extracted and saved, along with the HTML and JSON reports, in the SageMaker MLflow App. 

In [32]:
mlflow.set_experiment(mlflow_experiment_name)
timestamp_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')

with mlflow.start_run(run_name=f"data_drift_quality_monitoring_{timestamp_suffix}") as drift_run:
    drift_run_id = drift_run.info.run_id
    
    mlflow.log_params({
        'reference_data_size': len(reference_data),
        'current_data_size': len(production_data),
        'monitoring_timestamp': datetime.now().isoformat(),
        'model_name': model_name,
        'training_job': training_job_name
    })

    # Wrap DataFrames in Evidently Dataset objects
    drift_data_definition = DataDefinition()
    reference_dataset = Dataset.from_pandas(reference_data, data_definition=drift_data_definition)
    production_dataset = Dataset.from_pandas(production_data, data_definition=drift_data_definition)
    
    print('Creating Evidently data drift report...')
    data_drift_report = Report(metrics=[
        DataDriftPreset(),
    ])
    data_drift_report_snapshot = data_drift_report.run(
        reference_data=reference_dataset,
        current_data=production_dataset
    )
    
    data_drift_report_filename = f'data_drift_report_{timestamp_suffix}'
    data_drift_report_snapshot.save_html(f'reports/{data_drift_report_filename}.html')
    data_drift_report_snapshot.save_json(f'reports/{data_drift_report_filename}.json')
    
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.html', 'evidently_report_data_drift_html')
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.json', 'evidently_report_data_drift_json')
    
    drift_results = data_drift_report_snapshot.dict()
    log_metrics_from_dict(drift_results)

    print('Creating Evidently data summary report...')
    data_quality_report = Report(metrics=[
        DataSummaryPreset(),
    ])
    data_quality_report_snapshot = data_quality_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    data_quality_filename_html = f'data_quality_report_{timestamp_suffix}.html'
    data_quality_report_snapshot.save_html(f'reports/{data_quality_filename_html}')
    print(f'Data quality report saved to: {data_quality_filename_html}')
    mlflow.log_artifact(f'reports/{data_quality_filename_html}', 'evidently_report_data_quality')

[04/15/26 15:37:38] WARNING  Retrying (Retry(total=6, connect=7, read=6, redirect=7,          ]8;id=853942;file:///opt/conda/lib/python3.12/site-packages/urllib3/connectionpool.py\connectionpool.py]8;;\:]8;id=46081;file:///opt/conda/lib/python3.12/site-packages/urllib3/connectionpool.py#868\868]8;;\
                             status=7)) after connection broken by                                                 
                             'RemoteDisconnected('Remote end closed connection without                             
                             response')':                                                                          
                             /api/2.0/mlflow/experiments/get-by-name?experiment_name=demo-ml-                      
                             xgb-train-monitor                                                                     

Creating Evidently data drift report...
Creating Evidently data summary report...
Data quality report saved to: data_quality_report_20260415_153738.html
🏃 View run data_drift_quality_monitoring_20260415_153738 at: https://mlflow.sagemaker.us-west-2.app.aws/#/experiments/83/runs/0f0c96371a5d4fefbb846b02e1c41a10
🧪 View experiment at: https://mlflow.sagemaker.us-west-2.app.aws/#/experiments/83


The final report produced by Evidently can also be viewed directly inside this notebook.

In [33]:
from IPython.display import IFrame

print('Displaying Evidently Data Drift Report:')
IFrame(src=f'reports/{data_drift_report_filename}.html', width=1000, height=800)

Displaying Evidently Data Drift Report:


### View Drift Metrics in MLflow

To view your drift monitoring results in MLflow:

1. Open the SageMaker Studio MLflow UI
2. Navigate to the experiment: **demo-ml-xgb-train-monitor**
3. Find the run with name starting with **data_drift_**
4. View:
   - **Metrics**: drift scores for each feature, dataset-level drift
   - **Artifacts**: HTML reports and JSON summaries
   - **Parameters**: monitoring configuration and timestamps

---

## 7: Binary Classification Model Quality Evaluation with Evidently

Classification evaluation assesses how well your model performs on the prediction task by comparing predicted labels with actual ground truth labels.

### Key Classification Metrics

- **Accuracy**: Overall correctness of predictions
- **Precision**: Of all positive predictions, how many were actually positive (minimizes false positives)
- **Recall**: Of all actual positives, how many were correctly predicted (minimizes false negatives)
- **F1 Score**: Harmonic mean of precision and recall (balanced metric)
- **ROC-AUC**: Area under the ROC curve (measures discrimination ability)
- **Confusion Matrix**: Breakdown of true positives, true negatives, false positives, false negatives

### Why Evaluate Classification Performance?

- **Model Quality**: Understand model's predictive performance on test data
- **Business Metrics**: Align technical metrics with business objectives
- **Threshold Tuning**: Optimize decision threshold for your use case
- **Error Analysis**: Identify where the model makes mistakes

### Evidently Classification Features

- Comprehensive classification metrics (accuracy, precision, recall, F1, ROC-AUC)
- Confusion matrix visualization
- Classification quality by class
- Probability distribution analysis
- Prediction drift detection

First, merge the predictions produced by the real-time endpoint with the ground truth labels saved previously. These two datasets can be merged with the unique IDs.

In [34]:
gt_df = pd.read_csv('data/ground_truth.csv')
classification_eval_data = captured_df.merge(gt_df, on='inference_id', how='inner')

print(f'Merged dataset shape: {classification_eval_data.shape}')
print(f'  Captured records: {len(captured_df)}')
print(f'  Ground truth records: {len(gt_df)}')
print(f'  Matched records: {len(classification_eval_data)}')
print(f'\nActual labels distribution:')
print(classification_eval_data['target'].value_counts())
print(f'\nPredicted labels distribution:')
print(classification_eval_data['prediction'].value_counts())

Merged dataset shape: (6179, 24)
  Captured records: 6179
  Ground truth records: 6179
  Matched records: 6179

Actual labels distribution:
target
0    5483
1     696
Name: count, dtype: int64

Predicted labels distribution:
prediction
0    5589
1     590
Name: count, dtype: int64


Similar to data drift, Evidently has various presets for calculating different types of [model quality](https://docs.evidentlyai.com/metrics/preset_classification), all of which can be customized to suit the particular ML use case. In addition to saving the final report in MLflow, it can be helpful to extract specific quality values and save them separately as metrics in MLflow. This enables you to easily compare different runs or generate alerts based on certain values. The cell below contains a helper function which extracts certain metrics from the Evidently results to log in MLflow. It can be customized depending on the types of quality calculations you use.

In [35]:
import numpy as np

def log_evidently_classification_metrics_to_mlflow(metrics_dict):
    """Log Evidently classification metrics to MLflow using only metric prefix"""
    for metric in metrics_dict.get('metrics', []):
        # Extract only prefix before first '('
        metric_name_full = metric.get('metric_name', '')
        metric_name = metric_name_full.split('(')[0].strip()
        
        value = metric.get('value')
        
        if isinstance(value, dict):
            # Handle dict values like F1ByLabel, PrecisionByLabel, etc.
            for label, val in value.items():
                mlflow.log_metric(f"{metric_name}_label_{label}", float(val))
        else:
            # Handle scalar values (including np.float64)
            mlflow.log_metric(metric_name, float(value))

This sample uses the built-in `ClassificationPreset` from Evidently to calculate model quality from the inference results and the ground truth labels. The metrics are extracted and saved, along with the HTML and JSON reports, in the SageMaker MLflow App. 

In [36]:
with mlflow.start_run(run_name=f'model_quality_{timestamp_suffix}') as run:
    eval_data = classification_eval_data.copy()
    print(f"Evaluation dataset columns: {eval_data.columns.tolist()}")
    print(f"Evaluation dataset shape: {eval_data.shape}")
    
    # Create DataDefinition with BinaryClassification to tell Evidently which columns to use
    # This is REQUIRED for ClassificationPreset to work
    data_definition = DataDefinition(
        classification=[
            BinaryClassification(
                target='target',              # Column with actual labels
                prediction_labels='prediction', # Column with predicted labels
                pos_label=1,                  # Positive class label
            )
        ],
    )
    
    # Wrap the DataFrame in an Evidently Dataset
    eval_dataset = Dataset.from_pandas(
        eval_data,
        data_definition=data_definition,
    )
    
    print('\nCreating Evidently classification performance report...')
    classification_report = Report(metrics=[
        ClassificationPreset(),
    ])
    classification_report_snapshot = classification_report.run(
        reference_data=None,
        current_data=eval_dataset,
    )
    
    classification_report_filename = f'classification_report_{timestamp_suffix}'
    classification_report_snapshot.save_html(f'data/{classification_report_filename}.html')
    print(f'Classification report saved to: data/{classification_report_filename}.html')
    
    classification_report_snapshot.save_json(f'data/{classification_report_filename}.json')
    print(f'Classification report saved to: data/{classification_report_filename}.json')
    
    mlflow.log_artifact(f'data/{classification_report_filename}.html', 'evidently_classification_report_html')
    mlflow.log_artifact(f'data/{classification_report_filename}.json', 'evidently_classification_report_json')
    
    classification_report_dict = classification_report_snapshot.dict()
    log_evidently_classification_metrics_to_mlflow(classification_report_dict)

Evaluation dataset columns: ['inference_id', 'prediction_proba', 'prediction', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'target']
Evaluation dataset shape: (6179, 24)

Creating Evidently classification performance report...
Classification report saved to: data/classification_report_20260415_153738.html
Classification report saved to: data/classification_report_20260415_153738.json
🏃 View run model_quality_20260415_153738 at: https://mlflow.sagemaker.us-west-2.app.aws/#/experiments/83/runs/6c64097967e64cdaa038d313ba8a0145
🧪 View experiment at: https://mlflow.sagemaker.us-west-2.app.aws/#/experiments/83


### Classification Evaluation Summary

The classification report provides:

1. **Overall Performance Metrics**
   - Accuracy: Percentage of correct predictions
   - Precision: Quality of positive predictions
   - Recall: Coverage of actual positive cases
   - F1 Score: Balance between precision and recall
   - ROC-AUC: Discrimination ability across all thresholds

2. **Confusion Matrix**
   - True Positives (TP): Correctly predicted positive cases
   - True Negatives (TN): Correctly predicted negative cases
   - False Positives (FP): Incorrectly predicted as positive (Type I error)
   - False Negatives (FN): Incorrectly predicted as negative (Type II error)

3. **Class-Level Analysis**
   - Performance breakdown by class (0: No subscription, 1: Subscription)
   - Support: Number of samples per class
   - Per-class precision and recall

4. **Probability Calibration**
   - Distribution of predicted probabilities
   - Calibration curves showing reliability of probability estimates

### Business Impact

For the bank marketing use case:
- **High Precision**: Minimizes wasted effort on customers unlikely to subscribe
- **High Recall**: Maximizes capture of potential subscribers
- **F1 Score**: Balances both objectives for optimal campaign efficiency
- **Threshold Tuning**: Adjust decision boundary based on business costs/benefits

---

## 8: Comprehensive Monitoring Report (Optional)

Create a combined report with drift detection and data quality analysis.

In [37]:
with mlflow.start_run(run_name=f"comprehensive_monitoring_{timestamp_suffix}") as comp_run:
    print('Creating comprehensive monitoring report...')
    comprehensive_report = Report(metrics=[
        DataDriftPreset(),
        DataSummaryPreset(),
    ])
    comprehensive_report_snapshot = comprehensive_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    comp_filename = f'comprehensive_monitoring_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html'
    comprehensive_report_snapshot.save_html(comp_filename)
    print(f'Comprehensive report saved to: {comp_filename}')

    mlflow.log_artifact(comp_filename, 'evidently_comprehensive_report')
    comp_s3_key = f"{prefix}/evidently-reports/{comp_filename}"
    s3_client.upload_file(comp_filename, bucket, comp_s3_key)
    mlflow.log_param('comprehensive_report_s3', f's3://{bucket}/{comp_s3_key}')
    print(f'\nComprehensive monitoring report completed')
    print(f'Report location: s3://{bucket}/{comp_s3_key}')

Creating comprehensive monitoring report...
Comprehensive report saved to: comprehensive_monitoring_20260415_153755.html

Comprehensive monitoring report completed
Report location: s3://sagemaker-us-west-2-975049911976/bank-marketing-monitoring/evidently-reports/comprehensive_monitoring_20260415_153755.html
🏃 View run comprehensive_monitoring_20260415_153738 at: https://mlflow.sagemaker.us-west-2.app.aws/#/experiments/83/runs/17ff6a0ed3be4fc6aa60f84608703ac5
🧪 View experiment at: https://mlflow.sagemaker.us-west-2.app.aws/#/experiments/83


---

## 9: Summary and Next Steps

### What We Accomplished

1. **Model Training**: Trained XGBoost model using SageMaker SDK v3 with MLflow tracking
2. **Experiment Tracking**: Logged all parameters, metrics, and artifacts to SageMaker AI MLflow
3. **Real-Time Inference**: Deployed model to a SageMaker real-time endpoint with data capture enabled
4. **Data Capture**: Automatically captured request/response payloads to S3 for monitoring
5. **Classification Evaluation**: Evaluated binary classification performance using Evidently's ClassificationPreset with comprehensive metrics (accuracy, precision, recall, F1, ROC-AUC, confusion matrix)
6. **Drift Detection**: Used Evidently to detect data drift between training baseline and captured production data
7. **MLflow Integration**: Logged all classification metrics, drift metrics, and reports to MLflow for tracking and comparison
8. **Visual Reports**: Generated interactive HTML reports for classification performance, data drift, and quality analysis

### Next Steps

1. **Scaling**: Move the Evidently calculations into a processing job in a pipeline for repeated use at scale
2. **Automated Monitoring**: Use EventBridge rules to trigger monitoring when new captured data arrives in S3
3. **Alerting**: Configure SNS notifications for drift alerts and classification performance degradation
4. **Model Retraining**: Trigger retraining when drift exceeds thresholds using SageMaker Pipelines
5. **Enhanced Monitoring**: Monitor prediction distribution drift and business metrics over time
6. **Dashboard**: Create drift trend visualizations with MLflow UI

---

## 10: Cleanup (Optional)

Clean up resources to avoid unnecessary costs. **Important**: Real-time endpoints incur charges while running.

In [38]:
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Deleting endpoint: {endpoint_name} (this may take a minute)')
except Exception as e:
    print(f'Error deleting endpoint: {e}')

try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f'Deleted endpoint config')
except Exception as e:
    print(f'Error deleting endpoint config: {e}')

try:
    sm_client.delete_model(ModelName=model_name)
    print(f'Deleted model: {model_name}')
except Exception as e:
    print(f'Error deleting model: {e}')

print('\nNote: MLflow runs and S3 data are preserved for historical analysis.')

Error deleting endpoint: name 'sm_client' is not defined
Error deleting endpoint config: name 'sm_client' is not defined
Error deleting model: name 'sm_client' is not defined

Note: MLflow runs and S3 data are preserved for historical analysis.


---

## Additional Resources

### Documentation
- [SageMaker Python SDK v3](https://sagemaker.readthedocs.io/en/stable/)
- [SageMaker AI MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- [Evidently Documentation](https://docs.evidentlyai.com/)

### GitHub Repositories
- [Evidently GitHub](https://github.com/evidentlyai/evidently)
- [SageMaker Examples](https://github.com/aws/amazon-sagemaker-examples)